In [ ]:
import os, json

os.makedirs('/root/.kaggle', exist_ok=True)

kaggle_creds = {
    "username": "srivathsajr",
    "key": "KGAT_231faadfb84eafaac71bad1096d8dcbc"
}

with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_creds, f)

os.chmod('/root/.kaggle/kaggle.json', 600)

In [ ]:
!pip install kagglehub
import kagglehub
path = kagglehub.dataset_download("thoughtvector/customer-support-on-twitter")
print(path)

100%|██████████| 169M/169M [00:10<00:00, 16.1MB/s]

Extracting files...


/root/.cache/kagglehub/datasets/thoughtvector/customer-support-on-twitter/versions/10


In [ ]:
import os
files = os.listdir(path)
print(files)

['twcs', 'sample.csv']


In [ ]:
import os

twcs_folder = os.path.join(path, 'twcs')
print(os.listdir(twcs_folder))

['twcs.csv']


In [ ]:
import pandas as pd

csv_path = os.path.join(twcs_folder, 'twcs.csv')
df = pd.read_csv(csv_path)

print(df.shape)
print(df.columns.tolist())
df.head()

(2811774, 7)
['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0


In [ ]:
# find all unique brand-like author_ids (non-numeric ones are brands)
brands = df[~df['author_id'].str.isnumeric()]['author_id'].unique()
print(sorted(brands))

['ATT', 'ATVIAssist', 'AWSSupport', 'AdobeCare', 'AirAsiaSupport', 'AirbnbHelp', 'AlaskaAir', 'AldiUK', 'AmazonHelp', 'AmericanAir', 'AppleSupport', 'ArbysCares', 'ArgosHelpers', 'AskAmex', 'AskCiti', 'AskDSC', 'AskLyft', 'AskPapaJohns', 'AskPayPal', 'AskPlayStation', 'AskRBC', 'AskRobinhood', 'AskSeagate', 'AskTarget', 'AskTigogh', 'AskVirginMoney', 'Ask_Spectrum', 'Ask_WellsFargo', 'AskeBay', 'AsurionCares', 'AzureSupport', 'BofA_Help', 'BoostCare', 'British_Airways', 'CarlsJr', 'CenturyLinkHelp', 'ChaseSupport', 'ChipotleTweets', 'CoxHelp', 'DellCares', 'Delta', 'DoorDash_Help', 'DropboxSupport', 'DunkinDonuts', 'GWRHelp', 'GloCare', 'GoDaddyHelp', 'GooglePlayMusic', 'GreggsOfficial', 'HPSupport', 'HiltonHelp', 'HotelTonightCX', 'IHGService', 'JackBox', 'JetBlue', 'KFC_UKI_Help', 'KeyBank_Help', 'Kimpton', 'LondonMidland', 'MOO', 'MTNC_Care', 'McDonalds', 'MicrosoftHelps', 'Morrisons', 'NeweggService', 'NikeSupport', 'NortonSupport', 'O2', 'OPPOCareIN', 'OfficeSupport', 'PandoraSupp

In [ ]:
spotify_replies = df[(df['author_id'] == 'SpotifyCares') & (df['inbound'] == False)]
print(spotify_replies.shape)
spotify_replies.head()

(43265, 7)


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
540,848,SpotifyCares,False,Tue Oct 31 22:28:16 +0000 2017,@115887 Hmm. Can you try restarting your devic...,849,850.0
542,851,SpotifyCares,False,Tue Oct 31 23:39:03 +0000 2017,@115887 Could you send us a DM with your accou...,NaN,849.0
544,852,SpotifyCares,False,Tue Oct 31 21:04:13 +0000 2017,"@115887 Thanks. Just to be sure, are you Free ...",850,853.0
546,854,SpotifyCares,False,Tue Oct 31 19:36:16 +0000 2017,"@115887 Hey! What device, operating system, an...",853,855.0
548,856,SpotifyCares,False,Tue Oct 31 22:25:16 +0000 2017,@115889 Got it. It's not possible at the momen...,857,858.0


In [ ]:
# only keep customer_tweet_ids that actually exist in our dataset
valid_ids = customer_tweet_ids[customer_tweet_ids.isin(df_indexed.index)]

print(f"Total spotify replies with a linked ID: {len(customer_tweet_ids)}")
print(f"Valid IDs (actually found in dataset): {len(valid_ids)}")
print(f"Missing/dropped: {len(customer_tweet_ids) - len(valid_ids)}")

customer_tweets = df_indexed.loc[valid_ids]
print(customer_tweets.shape)

Total spotify replies with a linked ID: 43243
Valid IDs (actually found in dataset): 43206
Missing/dropped: 37
(43206, 6)


In [ ]:
# rebuild pairs by explicit merging, not positional matching
spotify_replies_valid = spotify_replies[spotify_replies['in_response_to_tweet_id'].isin(valid_ids)]

pairs = pd.DataFrame({
    'customer_tweet_id': spotify_replies_valid['in_response_to_tweet_id'].astype(int).values,
    'spotify_reply_id': spotify_replies_valid['tweet_id'].values,
    'spotify_reply_text': spotify_replies_valid['text'].values
})

# now attach the actual customer text by looking up each customer_tweet_id
pairs['customer_text'] = df_indexed.loc[pairs['customer_tweet_id'], 'text'].values

print(pairs.shape)
pairs.head(10)

(43206, 4)


,customer_tweet_id,spotify_reply_id,spotify_reply_text,customer_text
0,850,848,@115887 Hmm. Can you try restarting your devic...,@SpotifyCares Premium &amp; when i️ have it on...
1,849,851,@115887 Could you send us a DM with your accou...,@SpotifyCares doesn’t work and i even tried de...
2,853,852,"@115887 Thanks. Just to be sure, are you Free ...",@SpotifyCares iphone 7+ and i have the most re...
3,855,854,"@115887 Hey! What device, operating system, an...",i’m pissed my @115888 shuffle and repeat butto...
4,858,856,@115889 Got it. It's not possible at the momen...,@SpotifyCares 2/2... and there is no way to ma...
5,857,859,@115889 Sorry to hear that. The Spotify app on...,"@SpotifyCares Yes, multiple times. No changes...."
6,862,860,@115889 Hey there! That doesn't sound good. Wh...,@SpotifyCares @115890 Groove Music quits &amp;...
7,864,863,@115891 No worries. If you have other question...,@SpotifyCares ok thx
8,866,865,@115891 Hey Mikey! We're afraid there's no way...,is there a way to find non-explicit songs that...
9,1869,1867,@116128 Got it! Could you try streaming one of...,@SpotifyCares using a MacBook Pro with OS X El...


In [ ]:
sample = pairs.sample(n=40, random_state=42)
pd.set_option('display.max_colwidth', None)  # so we can see full text, not truncated
sample[['customer_text', 'spotify_reply_text']]

,customer_text,spotify_reply_text
2514,Can someone please tell me how to start an artist radio station o.n this @115888 app on this #xboxone app ? I be damned if I see how ??,"@169626 Hey there! At this time, we're afraid this isn't possible. There's more info about this here: https://t.co/IQvWN7FkZu /NQ"
29131,"@SpotifyCares having trouble activating a Family Premium account. Got the email with code but get 600 error message, now can't access page","@614298 Hey Matt, help's here! Can you DM us your account's username or email address? We'll take a look under the hood /RS https://t.co/ldFdZRiNAt"
23757,@554978 ただspotifyは無料会員だと、シャッフルでしか再生できなかったり途中で関係ない曲入れられたり早送りが５曲？くらいまでしかできなくて慣れるまで少しイライラします笑,@554977 ツイート拝見しました。Freeユーザーでスマホをご利用の場合はシャッフル再生となりますが、時間制限なしでお楽しみいただけます。タブレットやパソコンではシャッフルなしで15時間/30日までご利用いただけます。詳しくはこちら→ https://t.co/OoJIjpfeRy /AH
38178,"@SpotifyCares Thank you, I know that it’s something a lot of people would like too, there was even a petition started somewhere! X","@751180 We've already passed your feedback on to the right folks. If you need anything else, let us know. We're always... https://t.co/30FxtpFyVv 🙂 /BD"
35284,"@SpotifyCares It's not song specific, but it might be playlist specific. It doesn't happen on my Mac, and I only have this mobile device.",@707288 We understand. Could you send us a DM with your account's email address or username? We'll take a look backstage /CO https://t.co/ldFdZRiNAt
2531,@SpotifyCares https://t.co/t1VR724onN this is the playlist thx :),@169836 Got it! It looks like the songs in this playlist are not available. We have info about Spotify content here: https://t.co/0i8GpimuDa /PL
30450,@SpotifyCares I need to use an apo address but cant change country(apos are technically us) I don't want to do it via payment because it's to redeem a family account. Im moving but i really want to download my playlist before the flight.,@637757 Hey Ashlan! Can you DM us your account's email address? We'll take a look backstage /SJ https://t.co/ldFdZRiNAt
38226,@SpotifyCares why did u charge me twice and why won’t my Hulu work?!,@752488 Hey Lexi! We've just replied to your DM. Let's carry on chatting there /C
17636,@SpotifyCares i fixed the problem. i had not to include spotify app in the closed apps after lockscreen,@454711 Nice. Glad to hear it's all good. Just let us know if we can help with anything else. https://t.co/m4HWSbgHVZ /GK
29160,Hey @115888 I’m having trouble logging into my account and the websites not super helpful. Can you please DM me,"@614575 Hey Koshin, help's here! Can you DM us your account's username or email address? We'll take a look backstage /PL https://t.co/ldFdZRiNAt"


In [ ]:
sample_200 = pairs.sample(n=200, random_state=42)
sample_200 = sample_200.reset_index(drop=True)
sample_200.to_csv('sample_for_labeling.csv', index=False)

from google.colab import files
files.download('sample_for_labeling.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 7.5 MB/s eta 0:00:00


In [ ]:
from groq import Groq  # or whichever LLM API you're using

client = Groq(api_key="YOUR_GROQ_API_KEY_HEREIxGHtySpLSjgPsx09Y6WGdyb3FY1dy8UzFroTzzlfhdmA1JD8hD")

TAXONOMY_PROMPT = """
You are classifying a customer support tweet sent to SpotifyCares.

Classify the message into exactly one INTENT and one RESOLUTION_FLAG.

INTENT options:
- Account_Billing: login/password/account access, billing disputes, subscription changes, hacked accounts
- Technical_Troubleshooting: a bug, crash, playback issue, or device malfunction
- Feature_Request: wants something added or changed that doesn't exist yet
- General_Question: asks how something works or if something is possible, no personal issue
- Closure: confirming resolution or just thanking, no real ask

RESOLUTION_FLAG options:
- needs_dm: requires private account verification via DM
- needs_more_info: needs a diagnostic/clarifying question first
- answerable_now: answerable with public info, no follow-up needed
- already_closed: already resolved, just an acknowledgment

Respond in EXACTLY this format, nothing else:
INTENT: <one of the 5 options>
FLAG: <one of the 4 options>

Example:
Message: "I can't log into my account, tried everything"
INTENT: Account_Billing
FLAG: needs_dm

Now classify this message:
Message: "{message}"
"""

def classify_message(message):
    prompt = TAXONOMY_PROMPT.format(message=message)
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    text = response.choices[0].message.content

    intent = text.split("INTENT:")[1].split("\n")[0].strip()
    flag = text.split("FLAG:")[1].strip()

    return intent, flag

In [ ]:
print(classify_message("I can't log into my account, forgot my password"))

('Account_Billing', 'needs_dm')


In [ ]:
test_messages = [
    "My app keeps crashing every time I open a playlist",
    "Can you add lyrics back to the desktop app?",
    "Everything's working now, thanks for the help!",
    "How do I cancel my subscription?"
]

for msg in test_messages:
    print(msg, "->", classify_message(msg))

My app keeps crashing every time I open a playlist -> ('Technical_Troubleshooting', 'answerable_now')
Can you add lyrics back to the desktop app? -> ('Feature_Request', 'answerable_now')
Everything's working now, thanks for the help! -> ('Closure', 'already_closed')
How do I cancel my subscription? -> ('Account_Billing', 'answerable_now')


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving golden_set_labeling.xlsx to golden_set_labeling.xlsx


In [ ]:
import openpyxl
wb = openpyxl.load_workbook('golden_set_labeling.xlsx')  # match exactly what Colab printed
ws = wb['Labels']
print(ws.max_row)

201


In [ ]:
import time

correct_intent = 0
correct_flag = 0
total = 0
results = []

for row in range(2, 42):
    customer_text = ws.cell(row, 3).value
    true_intent = ws.cell(row, 5).value
    true_flag = ws.cell(row, 6).value

    pred_intent, pred_flag = classify_message(customer_text)

    results.append({
        'text': customer_text,
        'true_intent': true_intent, 'pred_intent': pred_intent,
        'true_flag': true_flag, 'pred_flag': pred_flag
    })

    if pred_intent == true_intent:
        correct_intent += 1
    if pred_flag == true_flag:
        correct_flag += 1
    total += 1

    time.sleep(0.5)

print(f"Intent accuracy: {correct_intent}/{total} = {correct_intent/total:.1%}")
print(f"Flag accuracy: {correct_flag}/{total} = {correct_flag/total:.1%}")

Intent accuracy: 32/40 = 80.0%
Flag accuracy: 32/40 = 80.0%


In [ ]:
print("=== INTENT MISMATCHES ===")
for r in results:
    if r['true_intent'] != r['pred_intent']:
        print(f"TEXT: {r['text'][:100]}")
        print(f"  TRUE: {r['true_intent']}  |  PREDICTED: {r['pred_intent']}")
        print()

print("=== FLAG MISMATCHES ===")
for r in results:
    if r['true_flag'] != r['pred_flag']:
        print(f"TEXT: {r['text'][:100]}")
        print(f"  TRUE: {r['true_flag']}  |  PREDICTED: {r['pred_flag']}")
        print()

=== INTENT MISMATCHES ===
TEXT: @SpotifyCares https://t.co/t1VR724onN this is the playlist thx :)
  TRUE: General_Question  |  PREDICTED: Closure

TEXT: Yooooo why is "Blessed" by @29428 and @17627 not on @115888?!?! 😭😭😭
  TRUE: Feature_Request  |  PREDICTED: General_Question

TEXT: @SpotifyCares No dice. Same problems.
  TRUE: Account_Billing  |  PREDICTED: Technical_Troubleshooting

TEXT: @SpotifyCares Do you require any of my account details to do this?
  TRUE: Account_Billing  |  PREDICTED: General_Question

TEXT: @127240 LISTEN I DON’T USE SPOTIFY THAT MUCH AND I DID SOMETHING THAT SKIPPED IT
  TRUE: General_Question  |  PREDICTED: Technical_Troubleshooting

TEXT: Did @115888 pull the radio feature option on songs?
  TRUE: Technical_Troubleshooting  |  PREDICTED: General_Question

TEXT: @SpotifyCares Your Windows Mobile app is still having issues, any word on getting the app updated?
  TRUE: Feature_Request  |  PREDICTED: Technical_Troubleshooting

TEXT: uhhhh @115888 I was listen

In [ ]:
from collections import Counter

# find the most common intent and flag across your golden set
intents_seen = [ws.cell(r, 5).value for r in range(2, 42)]
flags_seen = [ws.cell(r, 6).value for r in range(2, 42)]

most_common_intent = Counter(intents_seen).most_common(1)[0][0]
most_common_flag = Counter(flags_seen).most_common(1)[0][0]

print(f"Most common intent: {most_common_intent}")
print(f"Most common flag: {most_common_flag}")

trivial_intent_correct = sum(1 for r in results if r['true_intent'] == most_common_intent)
trivial_flag_correct = sum(1 for r in results if r['true_flag'] == most_common_flag)

print(f"Trivial baseline intent accuracy: {trivial_intent_correct}/40 = {trivial_intent_correct/40:.1%}")
print(f"Trivial baseline flag accuracy: {trivial_flag_correct}/40 = {trivial_flag_correct/40:.1%}")

Most common intent: Account_Billing
Most common flag: needs_dm
Trivial baseline intent accuracy: 16/40 = 40.0%
Trivial baseline flag accuracy: 18/40 = 45.0%


In [ ]:
def keyword_classify(text):
    text_lower = text.lower()

    # intent keywords, checked in priority order (same logic as our manual rules)
    if any(w in text_lower for w in ["thank", "thanks", "all good", "fixed", "working now"]):
        intent = "Closure"
    elif any(w in text_lower for w in ["account", "login", "log in", "password", "charge", "billing", "subscription", "hacked", "cancel"]):
        intent = "Account_Billing"
    elif any(w in text_lower for w in ["crash", "bug", "won't play", "wont play", "error", "glitch", "stopped", "broken"]):
        intent = "Technical_Troubleshooting"
    elif any(w in text_lower for w in ["please add", "bring back", "wish", "why isn't", "why is", "not on spotify"]):
        intent = "Feature_Request"
    else:
        intent = "General_Question"

    # flag keywords
    if any(w in text_lower for w in ["dm", "private message", "message us"]):
        flag = "needs_dm"
    elif "?" in text and intent in ["Technical_Troubleshooting", "Feature_Request"]:
        flag = "needs_more_info"
    elif intent == "Closure":
        flag = "already_closed"
    else:
        flag = "answerable_now"

    return intent, flag

keyword_intent_correct = 0
keyword_flag_correct = 0

for row in range(2, 42):
    customer_text = ws.cell(row, 3).value
    true_intent = ws.cell(row, 5).value
    true_flag = ws.cell(row, 6).value

    pred_intent, pred_flag = keyword_classify(customer_text)

    if pred_intent == true_intent:
        keyword_intent_correct += 1
    if pred_flag == true_flag:
        keyword_flag_correct += 1

print(f"Keyword baseline intent accuracy: {keyword_intent_correct}/40 = {keyword_intent_correct/40:.1%}")
print(f"Keyword baseline flag accuracy: {keyword_flag_correct}/40 = {keyword_flag_correct/40:.1%}")

Keyword baseline intent accuracy: 27/40 = 67.5%
Keyword baseline flag accuracy: 17/40 = 42.5%


In [ ]:
!pip install sentence-transformers

from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Build the pool: only rows where the resolution was genuinely informative
retrieval_pool = []
for row in range(2, ws.max_row + 1):
    intent = ws.cell(row, 5).value
    flag = ws.cell(row, 6).value
    if flag == "answerable_now":  # only real, grounded resolutions
        retrieval_pool.append({
            'customer_text': ws.cell(row, 3).value,
            'spotify_reply': ws.cell(row, 4).value,
            'intent': intent
        })

print(f"Retrieval pool size: {len(retrieval_pool)}")

# embed all customer messages in the pool, once
pool_texts = [item['customer_text'] for item in retrieval_pool]
pool_embeddings = embedder.encode(pool_texts)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Retrieval pool size: 75


In [ ]:
def retrieve_similar(new_message, top_k=1):
    new_embedding = embedder.encode([new_message])

    # cosine similarity between new message and every pool entry
    similarities = np.dot(pool_embeddings, new_embedding.T).flatten() / (
        np.linalg.norm(pool_embeddings, axis=1) * np.linalg.norm(new_embedding)
    )

    top_idx = np.argsort(similarities)[-top_k:][::-1]
    return [retrieval_pool[i] for i in top_idx]

In [ ]:
REPLY_PROMPT = """
You are drafting a customer support reply as SpotifyCares on Twitter.

A similar past case was resolved like this:
Customer said: "{past_customer}"
Spotify replied: "{past_reply}"

Now draft a reply to this NEW customer message, in a similar tone and style,
grounded in how the similar case was handled. Keep it short, like a real tweet reply.

New customer message: "{new_message}"

Reply:
"""

def draft_reply(new_message):
    similar = retrieve_similar(new_message, top_k=1)[0]
    prompt = REPLY_PROMPT.format(
        past_customer=similar['customer_text'],
        past_reply=similar['spotify_reply'],
        new_message=new_message
    )
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    return response.choices[0].message.content, similar

In [ ]:
reply, source = draft_reply("Why can't I find non-explicit versions of songs?")
print("DRAFTED REPLY:", reply)
print("\nGROUNDED ON:", source['customer_text'], "->", source['spotify_reply'])

DRAFTED REPLY: @yourhandle We have some info on why non‑explicit versions might not show up here: https://t.co/ABCdefG /JN

GROUNDED ON: @115888 I prefer #explicit songs, why can't I get an #implicit warning? -> @634683 We have some info about why this happens here: https://t.co/CAbaBnprCW /JN


In [ ]:
reply, source = draft_reply("My playlist keeps skipping songs randomly on my Samsung phone")
print("DRAFTED REPLY:", reply)
print("\nGROUNDED ON:", source['customer_text'], "->", source['spotify_reply'])

DRAFTED REPLY: @yourhandle Sure thing. In a nutshell, try clearing the Spotify app cache (Settings → Apps → Spotify → Storage) or reinstall the app, and double‑check Offline mode isn’t on. Keep us posted /MU

GROUNDED ON: @SpotifyCares I'll check on my computer when I have a minute. Maybe my phone's just bring fussy. (Also idk how to clear that on my phone's browser) -> @123421 Sure thing. In a nutshell, the guide says to download the Spotify app from the Roku Channel Store on your Roku device. Keep us posted /MU


In [ ]:
def handle_message(new_message):
    intent, flag = classify_message(new_message)

    if flag == "needs_dm":
        decision = "ESCALATE"
        reason = "Requires private account verification via DM; cannot be safely auto-resolved in public."
        reply = None
    elif flag == "needs_more_info":
        decision = "AUTO-HANDLE (ask follow-up)"
        reason = "Diagnostic info needed before resolution; safe to ask publicly."
        reply, _ = draft_reply(new_message)  # will likely draft a clarifying question, grounded on similar cases
    elif flag == "already_closed":
        decision = "AUTO-HANDLE (acknowledge)"
        reason = "Customer already resolved or closing out; simple acknowledgment suffices."
        reply = "Glad to hear it! Let us know if you need anything else. 🙂"
    else:  # answerable_now
        decision = "AUTO-HANDLE (respond)"
        reason = "Public information can resolve this without private data or escalation."
        reply, _ = draft_reply(new_message)

    return {
        'intent': intent,
        'flag': flag,
        'decision': decision,
        'reason': reason,
        'drafted_reply': reply
    }

In [ ]:
import json
print(json.dumps(handle_message("Why did you charge me twice this month?"), indent=2))

{
  "intent": "Account_Billing",
  "flag": "needs_dm",
  "decision": "ESCALATE",
  "reason": "Requires private account verification via DM; cannot be safely auto-resolved in public.",
  "drafted_reply": null
}


In [ ]:
def retrieve_similar_with_confidence(new_message, top_k=1, min_similarity=0.35):
    new_embedding = embedder.encode([new_message])
    similarities = np.dot(pool_embeddings, new_embedding.T).flatten() / (
        np.linalg.norm(pool_embeddings, axis=1) * np.linalg.norm(new_embedding)
    )
    top_idx = np.argsort(similarities)[-top_k:][::-1]
    best_score = similarities[top_idx[0]]

    if best_score < min_similarity:
        return None, best_score  # no confident match
    return retrieval_pool[top_idx[0]], best_score

def draft_reply_safe(new_message):
    match, score = retrieve_similar_with_confidence(new_message)
    if match is None:
        return None, f"No sufficiently similar precedent found (best score: {score:.2f}) — escalating instead of guessing."

    prompt = REPLY_PROMPT.format(
        past_customer=match['customer_text'],
        past_reply=match['spotify_reply'],
        new_message=new_message
    )
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    return response.choices[0].message.content, f"Grounded on similarity score {score:.2f}"

In [ ]:
reply, note = draft_reply_safe("My playlist keeps skipping songs randomly on my Samsung phone")
print("REPLY:", reply)
print("NOTE:", note)

REPLY: @YourHandle Got it. In a nutshell, try clearing the Spotify app cache or reinstalling the app on your Samsung phone, and double‑check Offline mode isn’t on. Keep us posted /MU
NOTE: Grounded on similarity score 0.54


In [ ]:
match, score = retrieve_similar_with_confidence("My playlist keeps skipping songs randomly on my Samsung phone")
print("MATCHED CUSTOMER TEXT:", match['customer_text'])
print("MATCHED REPLY:", match['spotify_reply'])
print("SCORE:", score)

MATCHED CUSTOMER TEXT: @SpotifyCares I'll check on my computer when I have a minute. Maybe my phone's just bring fussy. (Also idk how to clear that on my phone's browser)
MATCHED REPLY: @123421 Sure thing. In a nutshell, the guide says to download the Spotify app from the Roku Channel Store on your Roku device. Keep us posted /MU
SCORE: 0.53729683


In [ ]:
def is_actually_relevant(new_message, candidate):
    check_prompt = f"""
Are these two customer support messages about the same underlying issue or topic?
Answer with only YES or NO.

Message A: "{new_message}"
Message B: "{candidate['customer_text']}"
"""
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": check_prompt}],
        temperature=0
    )
    return "YES" in response.choices[0].message.content.upper()

def draft_reply_v2(new_message):
    match, score = retrieve_similar_with_confidence(new_message, min_similarity=0.0)  # get best match regardless

    if not is_actually_relevant(new_message, match):
        return None, f"Best retrieval match (score {score:.2f}) failed relevance check — escalating instead of guessing."

    prompt = REPLY_PROMPT.format(
        past_customer=match['customer_text'],
        past_reply=match['spotify_reply'],
        new_message=new_message
    )
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    return response.choices[0].message.content, f"Grounded and relevance-checked (retrieval score {score:.2f})"

In [ ]:
try:
    reply, note = draft_reply_v2("My playlist keeps skipping songs randomly on my Samsung phone")
    print("REPLY:", reply)
    print("NOTE:", note)
except Exception as e:
    print("ERROR:", e)

REPLY: @user Sure thing. In a nutshell, try clearing the Spotify app cache or reinstalling the app on your Samsung phone, and make sure you’re on the latest version and not in low‑data mode. Keep us posted /MU
NOTE: Grounded and relevance-checked (retrieval score 0.54)


In [ ]:
print(len(retrieval_pool))

75


In [ ]:
print(repr(client.api_key[:8]), "...", repr(client.api_key[-4:]), "length:", len(client.api_key))

'gsk_AIxG' ... 'D8hD' length: 56


In [ ]:
from groq import Groq
client = Groq(api_key="YOUR_GROQ_API_KEY_HEREIxGHtySpLSjgPsx09Y6WGdyb3FY1dy8UzFroTzzlfhdmA1JD8hD")
print(repr(client.api_key[:8]), "length:", len(client.api_key))

'gsk_AIxG' length: 56


In [ ]:
print(len(retrieval_pool))
print(ws.max_row)

75
201


In [ ]:
import time

judge_test_set = []

for row in range(50, 80):
    customer_text = ws.cell(row, 3).value
    actual_spotify_reply = ws.cell(row, 4).value

    result = handle_message(customer_text)

    judge_test_set.append({
        'row': row,
        'customer_text': customer_text,
        'actual_reply': actual_spotify_reply,
        'drafted_reply': result['drafted_reply'],
        'decision': result['decision'],
        'intent': result['intent'],
        'flag': result['flag']
    })
    time.sleep(0.5)

drafted = [r for r in judge_test_set if r['drafted_reply'] is not None]
print(f"Total: {len(judge_test_set)}, Drafted replies: {len(drafted)}, Escalated: {len(judge_test_set) - len(drafted)}")

Total: 30, Drafted replies: 25, Escalated: 5


In [ ]:
JUDGE_PROMPT = """
You are evaluating the quality of a drafted customer support reply for SpotifyCares.

Original customer message: "{customer}"
Drafted reply: "{reply}"

Score the reply on three criteria. For each, answer only PASS or FAIL.

1. RELEVANT: Does the reply actually address what the customer asked about?
2. APPROPRIATE: Is the tone right for a brand support account (helpful, not dismissive, not overly casual)?
3. SAFE: Does the reply avoid promising things support cannot deliver, avoid asking for sensitive info publicly, and avoid inventing specific facts (account details, dates, policies) it cannot know?

Respond in exactly this format:
RELEVANT: PASS or FAIL
APPROPRIATE: PASS or FAIL
SAFE: PASS or FAIL
"""

def judge_reply(customer, reply):
    prompt = JUDGE_PROMPT.format(customer=customer, reply=reply)
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    text = response.choices[0].message.content

    scores = {}
    for criterion in ['RELEVANT', 'APPROPRIATE', 'SAFE']:
        line = [l for l in text.split('\n') if criterion in l]
        scores[criterion] = 'PASS' if line and 'PASS' in line[0].upper() else 'FAIL'
    return scores

In [ ]:
for i, item in enumerate(drafted):
    print(f"--- #{i+1} ---")
    print(f"CUSTOMER: {item['customer_text'][:200]}")
    print(f"DRAFTED REPLY: {item['drafted_reply']}")
    print()

--- #1 ---
CUSTOMER: Hey @SpotifyCares, is it possible to know for how long a user has been using Spotify? TIA
DRAFTED REPLY: @{user_handle} Hey! You can check how long you’ve been with Spotify in your account stats here 👉 https://t.co/yourStatsLink. Take a look and ping us if you have any questions /BH https://t.co/extraImgLink

--- #2 ---
CUSTOMER: @SpotifyCares Thx :)
DRAFTED REPLY: Glad to hear it! Let us know if you need anything else. 🙂

--- #3 ---
CUSTOMER: Hey @115888 @116130 can we get the @407605 podcast on your platform please? Some of us don't use iTunes and it's pretty popular 🙏
DRAFTED REPLY: @YourHandle Hey! We’re working with the rights holders to bring the @407605 podcast to Spotify. Stay tuned! /ND

--- #4 ---
CUSTOMER: @SpotifyCares Halp what is this other device! Keeps pausing and switching over to it randomly during songs :( Win10Pro Build 15063.608 https://t.co/ynPqxmBvVn
DRAFTED REPLY: @yourhandle Hi! It looks like Spotify Connect is hopping to another device. Op

In [ ]:
for i, item in enumerate(drafted):
    s = item['judge_scores']
    print(f"{i+1}: {s['RELEVANT']} / {s['APPROPRIATE']} / {s['SAFE']}")

In [ ]:
import time

correct_intent = 0
correct_flag = 0
total = 0
full_results = []

for row in range(2, 202):
    customer_text = ws.cell(row, 3).value
    true_intent = ws.cell(row, 5).value
    true_flag = ws.cell(row, 6).value

    pred_intent, pred_flag = classify_message(customer_text)

    full_results.append({
        'row': row,
        'text': customer_text,
        'true_intent': true_intent, 'pred_intent': pred_intent,
        'true_flag': true_flag, 'pred_flag': pred_flag
    })

    if pred_intent == true_intent:
        correct_intent += 1
    if pred_flag == true_flag:
        correct_flag += 1
    total += 1

    if total % 20 == 0:
        print(f"Progress: {total}/200")
    time.sleep(0.5)

print(f"\nFinal intent accuracy: {correct_intent}/{total} = {correct_intent/total:.1%}")
print(f"Final flag accuracy: {correct_flag}/{total} = {correct_flag/total:.1%}")

Progress: 20/200
Progress: 40/200
Progress: 60/200
Progress: 80/200
Progress: 100/200
Progress: 120/200
Progress: 140/200
Progress: 160/200
Progress: 180/200
Progress: 200/200

Final intent accuracy: 159/200 = 79.5%
Final flag accuracy: 134/200 = 67.0%


In [ ]:
from collections import defaultdict

def precision_recall(results, key_true, key_pred):
    classes = set(r[key_true] for r in results) | set(r[key_pred] for r in results)
    stats = {}
    for c in classes:
        tp = sum(1 for r in results if r[key_true]==c and r[key_pred]==c)
        fp = sum(1 for r in results if r[key_true]!=c and r[key_pred]==c)
        fn = sum(1 for r in results if r[key_true]==c and r[key_pred]!=c)
        precision = tp/(tp+fp) if (tp+fp)>0 else 0
        recall = tp/(tp+fn) if (tp+fn)>0 else 0
        f1 = 2*precision*recall/(precision+recall) if (precision+recall)>0 else 0
        stats[c] = (tp, fp, fn, precision, recall, f1)
    return stats

print("=== INTENT: per-class precision/recall/F1 ===")
for c, (tp,fp,fn,p,r,f1) in precision_recall(full_results, 'true_intent', 'pred_intent').items():
    print(f"{c:<28} P={p:.2f}  R={r:.2f}  F1={f1:.2f}  (n_true={tp+fn})")

print("\n=== FLAG: per-class precision/recall/F1 ===")
for c, (tp,fp,fn,p,r,f1) in precision_recall(full_results, 'true_flag', 'pred_flag').items():
    print(f"{c:<20} P={p:.2f}  R={r:.2f}  F1={f1:.2f}  (n_true={tp+fn})")

=== INTENT: per-class precision/recall/F1 ===
Account_Billing              P=0.93  R=0.80  F1=0.86  (n_true=69)
General_Question             P=0.60  R=0.75  F1=0.67  (n_true=32)
Feature_Request              P=0.82  R=0.73  F1=0.77  (n_true=37)
Technical_Troubleshooting    P=0.78  R=0.82  F1=0.80  (n_true=49)
Closure                      P=0.76  R=1.00  F1=0.87  (n_true=13)

=== FLAG: per-class precision/recall/F1 ===
already_closed       P=0.79  R=0.85  F1=0.81  (n_true=13)
needs_dm             P=0.84  R=0.69  F1=0.76  (n_true=75)
answerable_now       P=0.62  R=0.80  F1=0.70  (n_true=75)
needs_more_info      P=0.41  R=0.30  F1=0.34  (n_true=37)
